# Chapter 7.6: Quantized Matmul Kernels — INT8, INT4, FP8

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain why quantized matmul differs** from standard matmul
2. **Implement INT8 matmul** with dequantization in Triton
3. **Understand INT4 packing/unpacking** and implement INT4 matmul
4. **Work with FP8 (E4M3/E5M2)** formats and their matmul kernels
5. **Implement per-group dequantization** in the kernel
6. **Compare W4A16, W8A8, W8A16** approaches
7. **Benchmark quantized vs full-precision** matmul performance

## Prerequisites

- Chapter 7.2 (Triton Programming)
- Basic understanding of quantization (Chapter on quantization)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.6_quantized_matmul.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.6_quantized_matmul.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Quantized Matmul is Different

### 1.1 Standard Matmul

In standard FP16 matmul: `C = A @ B` where all tensors are FP16.

### 1.2 Quantized Matmul

In quantized matmul, the weight matrix B is stored in lower precision (INT8/INT4/FP8). The kernel must **dequantize** during computation:

```
Standard:  C[i,j] = sum(A[i,k] * B[k,j])          -- FP16 × FP16
Quantized: C[i,j] = sum(A[i,k] * dequant(B_q[k,j])) -- FP16 × dequant(INT8)
```

Where `dequant(x) = x * scale + zero_point` (or just `x * scale` for symmetric quantization).

### 1.3 Why Not Just Dequantize First?

Dequantizing the entire weight matrix defeats the purpose of quantization:
- **Memory**: We'd need to store the full FP16 weight anyway
- **Bandwidth**: We'd read the full FP16 weight from memory

Instead, we dequantize **inside the kernel**, in registers/SRAM. The quantized weight is loaded from HBM (saving bandwidth), dequantized in registers (nearly free), and used for computation.

### 1.4 Quantization Formats Summary

| Format | Bits | Range | Used For | Models |
|--------|------|-------|----------|--------|
| FP16 | 16 | ±65504 | Baseline | All |
| BF16 | 16 | ±3.4e38 | Baseline | All modern |
| FP8 E4M3 | 8 | ±448 | Weights + activations | H100 native |
| INT8 | 8 | -128 to 127 | Weights, sometimes activations | GPTQ, SmoothQuant |
| INT4 | 4 | -8 to 7 | Weights only | AWQ, GPTQ-4bit |

In [ ]:
import torch
import numpy as np

# Visualize quantization formats
def compare_quantization_formats():
    """Show the bandwidth savings from different quantization formats."""
    
    # Typical LLaMA-7B linear layer
    in_features = 4096
    out_features = 11008  # FFN intermediate size
    
    formats = [
        ("FP16", 16, 1.0),
        ("FP8 (E4M3)", 8, 0.5),
        ("INT8", 8, 0.5),
        ("INT4", 4, 0.25),
        ("INT3", 3, 0.1875),
        ("INT2", 2, 0.125),
    ]
    
    weight_elements = in_features * out_features
    fp16_bytes = weight_elements * 2
    
    print(f"Weight matrix: {in_features} × {out_features} = {weight_elements:,} elements")
    print(f"{'Format':<15} {'Bits':>5} {'Weight Size':>12} {'vs FP16':>8} {'BW Savings':>11}")
    print("-" * 55)
    
    for name, bits, ratio in formats:
        size = weight_elements * bits / 8
        # Add scale overhead (per-group, group_size=128)
        group_size = 128
        num_groups = weight_elements / group_size
        scale_overhead = num_groups * 2  # FP16 scale per group
        total = size + scale_overhead
        
        savings = 1 - total / fp16_bytes
        print(f"{name:<15} {bits:>5} {total/1e6:>10.1f} MB {total/fp16_bytes:>7.1%} {savings:>10.0%}")
    
    print(f"\nFor LLaMA-7B ({6.7e9:.0e} params):")
    for name, bits, _ in formats:
        model_size = 6.7e9 * bits / 8 / 1e9
        print(f"  {name:<15}: {model_size:.1f} GB")

compare_quantization_formats()

## 2. INT8 Matmul with Dequantization

### 2.1 Symmetric Quantization

For symmetric quantization:
$$w_{float} = w_{int8} \times scale$$

Where `scale = max(|w|) / 127`.

### 2.2 Per-Channel vs Per-Group

- **Per-channel**: One scale per output channel → less overhead, lower accuracy
- **Per-group**: One scale per group of elements → more overhead, higher accuracy
- Typical group size: 128 elements

In [ ]:
import torch
import triton
import triton.language as tl

def quantize_int8_per_channel(weight):
    """
    Quantize a weight matrix to INT8 with per-channel (per-row) scales.
    
    Args:
        weight: (out_features, in_features) float tensor
    Returns:
        weight_int8: (out_features, in_features) int8 tensor
        scales: (out_features, 1) float tensor
    """
    # Find the maximum absolute value per row
    abs_max = weight.abs().max(dim=1, keepdim=True).values
    scales = abs_max / 127.0
    scales = scales.clamp(min=1e-10)  # Avoid division by zero
    
    # Quantize
    weight_int8 = torch.round(weight / scales).clamp(-128, 127).to(torch.int8)
    
    return weight_int8, scales


def quantize_int8_per_group(weight, group_size=128):
    """
    Quantize with per-group scales for higher accuracy.
    
    Each group of `group_size` elements along the K dimension
    shares one scale factor.
    """
    out_features, in_features = weight.shape
    assert in_features % group_size == 0
    
    # Reshape into groups
    weight_grouped = weight.reshape(out_features, -1, group_size)
    
    # Per-group max
    abs_max = weight_grouped.abs().max(dim=2, keepdim=True).values
    scales = abs_max / 127.0
    scales = scales.clamp(min=1e-10)
    
    # Quantize
    weight_int8 = torch.round(weight_grouped / scales).clamp(-128, 127).to(torch.int8)
    weight_int8 = weight_int8.reshape(out_features, in_features)
    scales = scales.reshape(out_features, -1)  # (out_features, in_features // group_size)
    
    return weight_int8, scales


if torch.cuda.is_available():
    # Test quantization accuracy
    weight = torch.randn(1024, 4096, device='cuda')
    
    w_int8_ch, scales_ch = quantize_int8_per_channel(weight)
    w_int8_gr, scales_gr = quantize_int8_per_group(weight, group_size=128)
    
    # Dequantize and measure error
    w_deq_ch = w_int8_ch.float() * scales_ch
    w_deq_gr = (w_int8_gr.float().reshape(1024, -1, 128) * 
                scales_gr.unsqueeze(-1)).reshape(1024, 4096)
    
    error_ch = (w_deq_ch - weight).abs().mean().item()
    error_gr = (w_deq_gr - weight).abs().mean().item()
    
    print(f"INT8 Quantization Error:")
    print(f"  Per-channel: {error_ch:.6f}")
    print(f"  Per-group-128: {error_gr:.6f}")
    print(f"  Improvement: {error_ch/error_gr:.1f}x")

In [ ]:
# INT8 Matmul Kernel in Triton

@triton.jit
def int8_matmul_kernel(
    # A is FP16: (M, K)
    # B is INT8: (K, N) with per-channel scales: (N,)
    a_ptr, b_ptr, scales_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    W8A16 matmul: FP16 activations × INT8 weights.
    
    Key difference from standard matmul:
    - Weight B is loaded as INT8 (half the memory bandwidth)
    - Dequantized to FP32 in registers before computation
    - Accumulation in FP32 for accuracy
    """
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)
    
    # Pointers to first tiles
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    
    # Load per-channel scales for the N columns we're computing
    scales = tl.load(scales_ptr + rn, mask=rn < N)
    
    # Accumulator in FP32
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k in range(0, K, BLOCK_K):
        # Load A tile (FP16 or FP32)
        a_mask = (rm[:, None] < M) & ((k + rk[None, :]) < K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0).to(tl.float32)
        
        # Load B tile (INT8!) — half the memory bandwidth!
        b_mask = ((k + rk[:, None]) < K) & (rn[None, :] < N)
        b_int8 = tl.load(b_ptrs, mask=b_mask, other=0)
        
        # Dequantize: b_float = b_int8 * scale
        # This happens in registers — essentially free!
        b_float = b_int8.to(tl.float32)  # INT8 → FP32
        # Note: per-channel scale is applied after accumulation for efficiency
        
        # Compute tile matmul
        acc += tl.dot(a, b_float)
        
        # Advance pointers
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    
    # Apply per-channel scales after accumulation
    acc = acc * scales[None, :]
    
    # Store result
    c_offsets = rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    tl.store(c_ptr + c_offsets, acc, mask=c_mask)


def int8_matmul(A, B_int8, scales):
    """W8A16 matmul: A (FP32) @ dequant(B_int8)."""
    M, K = A.shape
    K2, N = B_int8.shape
    assert K == K2
    
    C = torch.empty(M, N, device=A.device, dtype=torch.float32)
    
    BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
    grid = ((M + BLOCK_M - 1) // BLOCK_M, (N + BLOCK_N - 1) // BLOCK_N)
    
    int8_matmul_kernel[grid](
        A, B_int8, scales.squeeze(), C,
        M, N, K,
        A.stride(0), A.stride(1),
        B_int8.stride(0), B_int8.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return C


if torch.cuda.is_available():
    M, K, N = 256, 4096, 4096
    
    # Create test data
    A = torch.randn(M, K, device='cuda', dtype=torch.float32)
    W = torch.randn(N, K, device='cuda', dtype=torch.float32).T  # (K, N)
    
    # Quantize weights
    W_int8, scales = quantize_int8_per_channel(W.T)  # Per output channel
    W_int8 = W_int8.T.contiguous()  # Back to (K, N)
    scales = scales.squeeze()  # (N,)
    
    try:
        C_quant = int8_matmul(A, W_int8, scales)
        C_ref = A @ W
        
        error = (C_quant - C_ref).abs().mean().item()
        rel_error = error / C_ref.abs().mean().item()
        
        print(f"INT8 Matmul ({M}×{K} @ {K}×{N})")
        print(f"  Absolute error: {error:.6f}")
        print(f"  Relative error: {rel_error:.4%}")
    except Exception as e:
        print(f"Error: {e}")

## 3. INT4 Matmul: Packing, Unpacking, Dequantization

### 3.1 INT4 Packing

INT4 values are packed two per byte (or 8 per int32):

```
Original INT4 values: [3, -2, 7, -1, 0, 5, -8, 4]
Binary (4-bit each):  [0011, 1110, 0111, 1111, 0000, 0101, 1000, 0100]
Packed into 32-bit:   0100_1000_0101_0000_1111_0111_1110_0011
```

### 3.2 Unpacking in the Kernel

To extract INT4 values from a packed int32:

```c
// Extract the i-th 4-bit value from packed_val
int4_val = (packed_val >> (i * 4)) & 0xF;  // Unsigned [0, 15]
int4_signed = int4_val - 8;                 // Signed [-8, 7] (zero-point = 8)
float_val = (float)int4_signed * scale;     // Dequantize
```

In [ ]:
import torch
import numpy as np

def quantize_int4_per_group(weight, group_size=128):
    """
    Quantize weight to INT4 with per-group scales.
    
    INT4 range: [-8, 7] (signed) or [0, 15] (unsigned with zero_point=8)
    
    Returns:
        packed: (out_features, in_features // 8) int32 — 8 INT4 values per int32
        scales: (out_features, in_features // group_size) float16
        zeros: (out_features, in_features // group_size) int32 (packed zero points)
    """
    out_features, in_features = weight.shape
    assert in_features % group_size == 0
    assert in_features % 8 == 0  # For packing
    
    # Reshape into groups
    w_grouped = weight.reshape(out_features, -1, group_size)
    
    # Per-group statistics
    w_max = w_grouped.max(dim=2, keepdim=True).values
    w_min = w_grouped.min(dim=2, keepdim=True).values
    
    # Symmetric quantization to [-8, 7]
    abs_max = torch.maximum(w_max.abs(), w_min.abs())
    scales = abs_max / 7.0
    scales = scales.clamp(min=1e-10)
    
    # Quantize to [-8, 7]
    w_int4 = torch.round(w_grouped / scales).clamp(-8, 7).to(torch.int8)
    w_int4 = w_int4.reshape(out_features, in_features)
    
    # Convert to unsigned [0, 15] for packing
    w_uint4 = (w_int4 + 8).to(torch.uint8)
    
    # Pack: 8 INT4 values into one int32
    w_uint4 = w_uint4.reshape(out_features, -1, 8)  # Groups of 8
    packed = torch.zeros(out_features, in_features // 8, device=weight.device, dtype=torch.int32)
    for i in range(8):
        packed |= w_uint4[:, :, i].to(torch.int32) << (i * 4)
    
    scales = scales.reshape(out_features, -1)
    
    return packed, scales, w_int4


def dequantize_int4(packed, scales, group_size=128):
    """Dequantize INT4 packed weights."""
    out_features = packed.shape[0]
    in_features = packed.shape[1] * 8
    
    # Unpack: extract 8 INT4 values from each int32
    unpacked = torch.zeros(out_features, in_features, device=packed.device, dtype=torch.float32)
    for i in range(8):
        uint4_vals = (packed >> (i * 4)) & 0xF  # Extract 4 bits
        int4_vals = uint4_vals.float() - 8.0      # Convert to signed
        unpacked[:, i::8] = int4_vals
    
    # Apply scales
    unpacked = unpacked.reshape(out_features, -1, group_size)
    unpacked = unpacked * scales.unsqueeze(-1)
    unpacked = unpacked.reshape(out_features, in_features)
    
    return unpacked


if torch.cuda.is_available():
    weight = torch.randn(1024, 4096, device='cuda')
    packed, scales, w_int4 = quantize_int4_per_group(weight, group_size=128)
    
    print(f"Original weight: {weight.shape}, {weight.element_size()} bytes/elem")
    print(f"Packed INT4: {packed.shape}, {packed.element_size()} bytes/elem")
    print(f"Compression: {weight.numel() * 4 / (packed.numel() * 4):.1f}x")
    print(f"  (Plus {scales.numel() * 2} bytes for scales)")
    
    # Verify dequantization
    w_deq = dequantize_int4(packed, scales, group_size=128)
    error = (w_deq - weight).abs().mean().item()
    print(f"\nDequantization error: {error:.6f}")

## 4. FP8 Matmul

### 4.1 FP8 Formats

Two FP8 formats defined by the OFP specification:

| Format | Sign | Exponent | Mantissa | Range | Precision |
|--------|------|----------|----------|-------|----------|
| E4M3 | 1 | 4 | 3 | ±448 | Higher precision |
| E5M2 | 1 | 5 | 2 | ±57344 | Wider range |

**Typical usage:**
- **E4M3**: Weights and forward activations (need precision)
- **E5M2**: Gradients (need range for outliers)

### 4.2 FP8 Matmul on H100

The H100 has native FP8 tensor cores:
- `FP8 × FP8 → FP32` (with FP32 accumulation)
- **2x throughput** vs FP16 (1979 TFLOPS vs 990 TFLOPS)
- Per-tensor or per-block scaling for accuracy

In [ ]:
# FP8 simulation (actual FP8 requires H100 + recent PyTorch)

def simulate_fp8_e4m3(x):
    """
    Simulate FP8 E4M3 quantization.
    
    E4M3: 1 sign + 4 exponent + 3 mantissa bits
    Range: [-448, 448]
    Smallest positive: 2^-9 ≈ 0.00195
    Precision: 3 mantissa bits → ~1/8 = 12.5% relative error
    """
    # Clamp to FP8 E4M3 range
    x_clamped = x.clamp(-448, 448)
    
    # Simulate reduced precision (3 mantissa bits = 8 levels per power of 2)
    # For each value, round to nearest representable FP8 value
    sign = x_clamped.sign()
    abs_x = x_clamped.abs().clamp(min=2**-9)  # Smallest positive
    
    # Find the exponent
    exp = torch.floor(torch.log2(abs_x))
    
    # Quantize mantissa to 3 bits (8 levels)
    mantissa = abs_x / (2.0 ** exp)  # Normalize to [1, 2)
    mantissa_quantized = torch.round(mantissa * 8) / 8  # 3-bit quantization
    
    result = sign * mantissa_quantized * (2.0 ** exp)
    return result


if torch.cuda.is_available():
    # Compare FP8 vs FP16 precision
    x = torch.randn(1000, device='cuda') * 10  # Scale to test range
    
    x_fp8 = simulate_fp8_e4m3(x)
    
    error = (x_fp8 - x).abs()
    rel_error = error / x.abs().clamp(min=1e-10)
    
    print(f"FP8 E4M3 Simulation:")
    print(f"  Mean absolute error: {error.mean():.6f}")
    print(f"  Mean relative error: {rel_error.mean():.4%}")
    print(f"  Max relative error:  {rel_error.max():.4%}")
    print()
    
    # Show precision at different magnitudes
    for mag in [0.01, 0.1, 1.0, 10.0, 100.0]:
        x_test = torch.tensor([mag], device='cuda')
        x_fp8_test = simulate_fp8_e4m3(x_test)
        print(f"  {mag:8.2f} → {x_fp8_test.item():8.4f} (error: {abs(x_fp8_test.item() - mag):.6f})")

## 5. W4A16 vs W8A8 vs W8A16 Comparison

### 5.1 Notation

- **W4A16**: 4-bit weights, 16-bit activations
- **W8A8**: 8-bit weights, 8-bit activations  
- **W8A16**: 8-bit weights, 16-bit activations

### 5.2 Trade-offs

| Method | Weight Size | Activation Size | Dequant Cost | Accuracy | Speed |
|--------|-----------|----------------|--------------|----------|-------|
| FP16 | 2 bytes | 2 bytes | None | Best | Baseline |
| W8A16 | 1 byte | 2 bytes | Low | Very Good | ~1.5x |
| W8A8 | 1 byte | 1 byte | Medium | Good | ~2x (INT8 TC) |
| W4A16 | 0.5 byte | 2 bytes | Higher | Varies | ~2-3x |

### 5.3 When to Use Which

- **W4A16 (AWQ, GPTQ)**: When model doesn't fit in GPU memory
- **W8A8 (SmoothQuant)**: Best speed with acceptable accuracy, needs calibration
- **W8A16**: Good balance, easy to apply, minimal accuracy loss
- **FP8 (H100)**: Best of both worlds on Hopper GPUs

In [ ]:
# Comprehensive benchmark framework
import time

def benchmark_quantized_matmul():
    """Compare performance of different quantization methods."""
    if not torch.cuda.is_available():
        print("GPU required")
        return
    
    M = 256  # Batch size
    K = 4096  # Input features
    N = 4096  # Output features
    n_iter = 100
    
    print(f"Matmul benchmark: ({M}, {K}) @ ({K}, {N})")
    print("=" * 60)
    
    # FP32 baseline
    A_fp32 = torch.randn(M, K, device='cuda', dtype=torch.float32)
    W_fp32 = torch.randn(K, N, device='cuda', dtype=torch.float32)
    
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        _ = torch.mm(A_fp32, W_fp32)
    torch.cuda.synchronize()
    fp32_time = (time.perf_counter() - start) / n_iter * 1e6
    
    # FP16
    A_fp16 = A_fp32.half()
    W_fp16 = W_fp32.half()
    
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        _ = torch.mm(A_fp16, W_fp16)
    torch.cuda.synchronize()
    fp16_time = (time.perf_counter() - start) / n_iter * 1e6
    
    # Calculate metrics
    flops = 2 * M * N * K
    weight_bytes_fp16 = K * N * 2
    weight_bytes_int8 = K * N * 1
    weight_bytes_int4 = K * N * 0.5
    
    print(f"{'Method':<15} {'Time (µs)':>10} {'Speedup':>8} {'Weight Mem':>12} {'BW Savings':>12}")
    print("-" * 60)
    print(f"{'FP32':<15} {fp32_time:>10.1f} {'1.00x':>8} {K*N*4/1e6:>10.1f} MB {'0%':>12}")
    print(f"{'FP16':<15} {fp16_time:>10.1f} {fp32_time/fp16_time:>7.2f}x {weight_bytes_fp16/1e6:>10.1f} MB {'50%':>12}")
    print(f"{'W8A16 (est.)':<15} {fp16_time*0.7:>10.1f} {fp32_time/(fp16_time*0.7):>7.2f}x {weight_bytes_int8/1e6:>10.1f} MB {'75%':>12}")
    print(f"{'W4A16 (est.)':<15} {fp16_time*0.5:>10.1f} {fp32_time/(fp16_time*0.5):>7.2f}x {weight_bytes_int4/1e6:>10.1f} MB {'87.5%':>12}")
    print()
    print("Note: W8A16 and W4A16 times are estimated based on bandwidth reduction.")
    print("Actual speedups depend on dequantization overhead and kernel quality.")

benchmark_quantized_matmul()

## 6. Marlin Kernel: State-of-the-Art INT4 Matmul

### 6.1 What Makes Marlin Fast?

Marlin is the fastest INT4 matmul kernel for NVIDIA GPUs. Key optimizations:

1. **Asynchronous Global-to-Shared Memory Copies** (cp.async)
2. **Register-Level Dequantization** — unpack INT4 in registers, avoid shared memory
3. **Tensor Core Friendly Layout** — data layout optimized for WMMA/MMA
4. **Double Buffering** — overlap loads with computation
5. **Fine-Grained Scheduling** — careful instruction ordering to hide latency

### 6.2 Performance

Marlin achieves:
- **~4x speedup** over FP16 matmul for weight-only quantization (batch=1)
- Near-ideal bandwidth utilization (>80% of peak HBM bandwidth)
- Used in vLLM for AWQ and GPTQ INT4 inference

In [ ]:
# Explain the Marlin data layout
def explain_marlin_layout():
    print("Marlin INT4 Kernel — Data Layout")
    print("=" * 60)
    print()
    print("Standard INT4 packing: 8 values per int32, row-major")
    print("  [v0 v1 v2 v3 v4 v5 v6 v7] → one int32")
    print()
    print("Marlin layout: Reordered for tensor core access patterns")
    print("  - 16×16 tiles aligned with MMA instruction operands")
    print("  - INT4 values pre-shuffled for register-level unpacking")
    print("  - Scales interleaved with quantized data for cache locality")
    print()
    print("Pipeline:")
    print("  1. cp.async: Load INT4 data from HBM → shared memory")
    print("  2. In registers: Unpack INT4 → FP16 (bit manipulation)")
    print("  3. Tensor cores: FP16 × FP16 → FP32 accumulation")
    print("  4. Apply group scales")
    print()
    print("Key insight: The dequantization (INT4 → FP16) happens")
    print("entirely in registers, overlapped with memory loads.")
    print("This makes it essentially free!")

explain_marlin_layout()

## 7. Per-Group Dequantization in the Kernel

### 7.1 Challenge

With per-group quantization, each group of elements along the K dimension has its own scale. This means:
- The scale changes as we iterate over K tiles
- We need to look up the correct scale for each K position
- This adds complexity to the inner loop

In [ ]:
@triton.jit
def int8_matmul_pergroup_kernel(
    a_ptr, b_ptr, scales_ptr, c_ptr,
    M, N, K,
    group_size,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_sn, stride_sg,  # scales: (N, K // group_size)
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    INT8 matmul with per-group dequantization.
    
    Each group of `group_size` elements along K shares one scale.
    The kernel must look up the correct scale as it iterates over K tiles.
    """
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)
    
    a_ptrs = a_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    b_ptrs = b_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn
    
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k in range(0, K, BLOCK_K):
        # Load A tile
        a_mask = (rm[:, None] < M) & ((k + rk[None, :]) < K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0).to(tl.float32)
        
        # Load B tile (INT8)
        b_mask = ((k + rk[:, None]) < K) & (rn[None, :] < N)
        b_int8 = tl.load(b_ptrs, mask=b_mask, other=0).to(tl.float32)
        
        # Load per-group scales for the current K position
        # group_idx = k // group_size (which group does this K tile belong to?)
        group_idx = k // group_size
        scales = tl.load(
            scales_ptr + rn * stride_sn + group_idx * stride_sg,
            mask=rn < N
        )
        
        # Dequantize: apply per-group scale to each column
        b_deq = b_int8 * scales[None, :]  # Broadcast scale over K dim
        
        # Compute
        acc += tl.dot(a, b_deq)
        
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    
    c_offsets = rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    tl.store(c_ptr + c_offsets, acc, mask=c_mask)

print("Per-group INT8 matmul kernel defined.")
print("Key change: scales are looked up per K-tile, not per-channel.")

## 8. Summary

| Topic | Key Insight |
|-------|------------|
| **Why in-kernel dequant** | Load quantized (low BW), dequant in registers (free) |
| **INT8** | 2x bandwidth savings, <1% accuracy loss |
| **INT4** | 4x bandwidth savings, needs careful group quantization |
| **FP8** | Native H100 support, 2x tensor core throughput |
| **Per-group scales** | Better accuracy, scale lookup adds complexity |
| **Marlin** | SOTA INT4 kernel: cp.async + register dequant + tensor cores |

### In vLLM/SGLang

| Quantization | Backend | Method |
|-------------|---------|--------|
| GPTQ (INT4) | Marlin, ExLlama | W4A16 per-group |
| AWQ (INT4) | Marlin, GEMM | W4A16 per-group |
| SmoothQuant (INT8) | cutlass INT8 | W8A8 per-tensor |
| FP8 | cutlass FP8 | W8A8 per-tensor |

## Exercises

### Exercise 1: Implement an INT8 W8A8 Matmul

Both activations AND weights are INT8. You need:
- Quantize activations online (per-token scaling)
- INT8 × INT8 → INT32 accumulation
- Dequantize the output

### Exercise 2: Mixed-Precision Accumulation

Implement a matmul where computation uses FP16 but accumulation uses FP32 to prevent overflow.

### Exercise 3: Benchmark Group Size Impact

Measure accuracy and speed for different group sizes (32, 64, 128, 256) in INT4 quantization.

In [ ]:
# Exercise 3: Measure group size impact
if torch.cuda.is_available():
    weight = torch.randn(4096, 4096, device='cuda')
    
    print(f"{'Group Size':>10} | {'Mean Error':>12} | {'Max Error':>12} | {'Scale Overhead':>15}")
    print("-" * 55)
    
    for group_size in [32, 64, 128, 256, 512, 1024]:
        if weight.shape[1] % group_size != 0:
            continue
        packed, scales, _ = quantize_int4_per_group(weight, group_size=group_size)
        deq = dequantize_int4(packed, scales, group_size=group_size)
        
        mean_err = (deq - weight).abs().mean().item()
        max_err = (deq - weight).abs().max().item()
        scale_bytes = scales.numel() * 2  # FP16 scales
        
        print(f"{group_size:>10} | {mean_err:>12.6f} | {max_err:>12.6f} | {scale_bytes/1024:>12.1f} KB")